##### Keras Tuner - Decide the number of hidden layers and the number of hidden neurons

In [1]:
import pandas as pd
import numpy as np
from tensorflow import keras
from tensorflow.keras import layers

from kerastuner import HyperModel, RandomSearch


C:\Users\atalb\AppData\Local\Temp\ipykernel_5988\2771968064.py:6: DeprecationWarning: `import kerastuner` is deprecated, please use `import keras_tuner`.
  from kerastuner import HyperModel, RandomSearch


In [ ]:
## Air quality dataset
dataset_path = r"C:\Users\atalb\Documents\Coding\DeepLearning\Datasets\Real_Combine.csv"
df = pd.read_csv(dataset_path)

In [3]:
df.head()

,T,TM,Tm,SLP,H,VV,V,VM,PM 2.5
0,7.4,9.8,4.8,1017.6,93.0,0.5,4.3,9.4,219.720833
1,7.8,12.7,4.4,1018.5,87.0,0.6,4.4,11.1,182.187500
2,6.7,13.4,2.4,1019.4,82.0,0.6,4.8,11.1,154.037500
3,8.6,15.5,3.3,1018.7,72.0,0.8,8.1,20.6,223.208333
4,12.4,20.9,4.4,1017.3,61.0,1.3,8.7,22.2,200.645833


In [4]:
X = df.iloc[:,:-1]
y = df.iloc[:,-1]

In [5]:
def build_model(hp):
    model = keras.Sequential()
    for i in range(hp.Int('num_layers', 2, 20)):
        model.add(layers.Dense(units=hp.Int('units_' + str(i), min_value=32, max_value=512, step=32),
                               activation='relu'))
    model.add(layers.Dense(1, activation='linear'))
    model.compile(optimizer=keras.optimizers.Adam(hp.Choice('learning_rate', [1e-2, 1e-3, 1e-4])),
                    loss='mean_absolute_error',
                    metrics=['mean_absolute_error'])
    return model

## hyperparameters
1. How many number of hidden layers
2. How many number of hidden neurons we should have in hidden layers
3. Learning rate

In [6]:
tuner = RandomSearch(
    build_model,
    objective='val_mean_absolute_error',
    max_trials=5,
    executions_per_trial=3,
    directory='project_1',
    project_name='air_quality_index'
)

In [7]:
tuner.search_space_summary()

Search space summary
Default search space size: 4
num_layers (Int)
{'default': None, 'conditions': [], 'min_value': 2, 'max_value': 20, 'step': 1, 'sampling': 'linear'}
units_0 (Int)
{'default': None, 'conditions': [], 'min_value': 32, 'max_value': 512, 'step': 32, 'sampling': 'linear'}
units_1 (Int)
{'default': None, 'conditions': [], 'min_value': 32, 'max_value': 512, 'step': 32, 'sampling': 'linear'}
learning_rate (Choice)
{'default': 0.01, 'conditions': [], 'values': [0.01, 0.001, 0.0001], 'ordered': True}


In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

In [11]:
tuner.search(X_train, y_train, epochs=5, validation_data=(X_test, y_test))

Epoch 1/5
24/24 ━━━━━━━━━━━━━━━━━━━━ 2s 22ms/step - loss: 72.5198 - mean_absolute_error: 72.5198 - val_loss: nan - val_mean_absolute_error: nan
Epoch 2/5
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 66.5989 - mean_absolute_error: 66.5989 - val_loss: nan - val_mean_absolute_error: nan
Epoch 3/5
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 63.8629 - mean_absolute_error: 63.8629 - val_loss: nan - val_mean_absolute_error: nan
Epoch 4/5
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 61.3920 - mean_absolute_error: 61.3920 - val_loss: nan - val_mean_absolute_error: nan
Epoch 5/5
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 10ms/step - loss: 59.9320 - mean_absolute_error: 59.9320 - val_loss: nan - val_mean_absolute_error: nan
Epoch 1/5
24/24 ━━━━━━━━━━━━━━━━━━━━ 3s 22ms/step - loss: 73.4636 - mean_absolute_error: 73.4636 - val_loss: nan - val_mean_absolute_error: nan
Epoch 2/5
24/24 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step - loss: 65.6551 - mean_absolute_error: 65.6551 - val_loss: nan - val_mean_absolute_err

c:\Users\atalb\Documents\Coding\DeepLearning\.venv\Lib\site-packages\keras_tuner\src\engine\metrics_tracking.py:111: RuntimeWarning: All-NaN axis encountered
  np.nanmin(values) if self.direction == "min" else np.nanmax(values)


RuntimeError: Number of consecutive failures exceeded the limit of 3.
Traceback (most recent call last):
  File "c:\Users\atalb\Documents\Coding\DeepLearning\.venv\Lib\site-packages\keras_tuner\src\engine\base_tuner.py", line 274, in _try_run_and_update_trial
    self._run_and_update_trial(trial, *fit_args, **fit_kwargs)
  File "c:\Users\atalb\Documents\Coding\DeepLearning\.venv\Lib\site-packages\keras_tuner\src\engine\base_tuner.py", line 239, in _run_and_update_trial
    results = self.run_trial(trial, *fit_args, **fit_kwargs)
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\atalb\Documents\Coding\DeepLearning\.venv\Lib\site-packages\keras_tuner\src\engine\tuner.py", line 309, in run_trial
    self._configure_tensorboard_dir(callbacks, trial, execution)
  File "c:\Users\atalb\Documents\Coding\DeepLearning\.venv\Lib\site-packages\keras_tuner\src\engine\tuner.py", line 421, in _configure_tensorboard_dir
    from tensorboard.plugins.hparams import api as hpa